In [ ]:
import pandas as pd
import ast
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer

df = pd.read_csv("disease_features.csv")
df_onehot = pd.read_csv("encoded_output2.csv")

def parse_and_join(column):
    return df[column].apply(lambda x: " ".join(ast.literal_eval(x)) if pd.notna(x) and x.strip() != "[]" else "")

df['Risk_Factors_str'] = parse_and_join('Risk Factors')
df['Symptoms_str'] = parse_and_join('Symptoms')
df['Signs_str'] = parse_and_join('Signs')

tfidf_risk = TfidfVectorizer()
tfidf_symptoms = TfidfVectorizer()
tfidf_signs = TfidfVectorizer()

X_risk = tfidf_risk.fit_transform(df['Risk_Factors_str'])
X_symptoms = tfidf_symptoms.fit_transform(df['Symptoms_str'])
X_signs = tfidf_signs.fit_transform(df['Signs_str'])

X_tfidf_combined = sp.hstack([X_risk, X_symptoms, X_signs])
X_onehot = df_onehot.drop(columns=['Disease'])
X_onehot_sparse = sp.csr_matrix(X_onehot.values)

In [ ]:
from sklearn.decomposition import PCA, TruncatedSVD
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.cm as cm

df['Subtype_Label'] = df['Subtypes'].apply(lambda x: list(ast.literal_eval(x).keys())[0] if pd.notna(x) else "Unknown")
labels = df['Subtype_Label'].values
unique_labels = list(set(labels))

def plot_2d(X, title):
    colors = cm.tab10(np.linspace(0, 1, len(unique_labels)))
    label_to_color = {label: colors[i] for i, label in enumerate(unique_labels)}
    plt.figure(figsize=(8, 6))
    for label in unique_labels:
        idx = np.array(labels) == label
        plt.scatter(X[idx, 0], X[idx, 1], label=label, alpha=0.7)
    plt.title(title)
    plt.xlabel("Component 1")
    plt.ylabel("Component 2")
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

plot_2d(PCA(n_components=2).fit_transform(X_tfidf_combined.toarray()), "PCA - TF-IDF")
plot_2d(PCA(n_components=2).fit_transform(X_onehot_sparse.toarray()), "PCA - One-Hot")
plot_2d(TruncatedSVD(n_components=2).fit_transform(X_tfidf_combined), "SVD - TF-IDF")
plot_2d(TruncatedSVD(n_components=2).fit_transform(X_onehot_sparse), "SVD - One-Hot")

In [ ]:
from sklearn.model_selection import LeaveOneOut
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

y = LabelEncoder().fit_transform(df['Subtype_Label'])
loo = LeaveOneOut()

# KNN (k=5, Euclidean)
knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
y_preds, y_trues = [], []
for train_idx, test_idx in loo.split(X_tfidf_combined):
    knn.fit(X_tfidf_combined[train_idx], y[train_idx])
    y_pred = knn.predict(X_tfidf_combined[test_idx])
    y_preds.append(y_pred[0])
    y_trues.append(y[test_idx][0])

print("KNN - TFIDF Accuracy:", accuracy_score(y_trues, y_preds))
print("KNN - TFIDF F1 Score:", f1_score(y_trues, y_preds, average='macro', zero_division=0))

In [ ]:
# Logistic Regression on SVD-reduced
X_tfidf_svd = TruncatedSVD(n_components=15).fit_transform(X_tfidf_combined)
logreg = LogisticRegression(solver='liblinear')
y_preds_lr, y_trues_lr = [], []

for train_idx, test_idx in loo.split(X_tfidf_svd):
    logreg.fit(X_tfidf_svd[train_idx], y[train_idx])
    y_pred = logreg.predict(X_tfidf_svd[test_idx])
    y_preds_lr.append(y_pred[0])
    y_trues_lr.append(y[test_idx][0])

print("Logistic Regression - TFIDF (SVD) Accuracy:", accuracy_score(y_trues_lr, y_preds_lr))
print("Logistic Regression - TFIDF (SVD) F1 Score:", f1_score(y_trues_lr, y_preds_lr, average='macro', zero_division=0))